# 07. Tuning Random Forest + CSL (Random Search)

Notebook ini melakukan **Randomized Search** untuk hyperparameter **Random Forest + cost-sensitive learning** menggunakan **Macro F1** dengan 5-fold cross-validation.

Data yang dipakai hanya `train_final.csv`. Isi dan logika mengikuti script sumber `tune_random_forest_final.py`.

## 0. Mengatur Folder Kerja

Notebook ini disimpan di folder `notebooks`, sedangkan data dan artifacts berada di folder `glysense_backend`. Sel berikut memindahkan folder kerja satu tingkat ke atas agar jalur seperti `data/...` dan `artifacts/...` ditemukan. Aman dijalankan berulang.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

## 1. Membaca Data Train Final

Jalur file mengasumsikan notebook dijalankan dari direktori utama `glysense_backend`.

In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score, make_scorer

train_path = "data/final_split/train_final.csv"

df_train = pd.read_csv(train_path)

## 2. Memisahkan Fitur dan Target

In [ ]:
X_train = df_train.drop(columns=["Diabetes_012"])

y_train = df_train["Diabetes_012"].astype(int)

## 3. Membuat Scorer Macro F1

In [ ]:
macro_f1_scorer = make_scorer(
    f1_score,
    average="macro",
    zero_division=0
)

## 4. Model Dasar Random Forest + CSL

In [ ]:
rf_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

## 5. Ruang Pencarian Hyperparameter

In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 5, 10, 15, 20, 30],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

## 6. Konfigurasi Random Search

In [ ]:
random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring=macro_f1_scorer,
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

## 7. Menjalankan Tuning

Sel ini memerlukan waktu cukup lama (20 kandidat × 5 fold).

In [ ]:
print("=" * 75)
print("RANDOM SEARCH FINAL RANDOM FOREST + CSL")
print("=" * 75)

random_search.fit(
    X_train,
    y_train
)

## 8. Menampilkan Hasil Terbaik

In [ ]:
print("\n" + "=" * 75)
print("HASIL HYPERPARAMETER TERBAIK")
print("=" * 75)

print("\nBest Parameters:")
print(random_search.best_params_)

print("\nBest Macro F1 Cross-Validation:")
print(round(random_search.best_score_, 4))

## 9. Menyimpan Hasil Parameter

Sel ini menimpa `artifacts/random_forest_best_params.txt` jika file sudah ada. Folder `artifacts` harus sudah ada.

In [ ]:
result_path = "artifacts/random_forest_best_params.txt"

with open(result_path, "w") as file:
    file.write("Best Parameters:\n")
    file.write(str(random_search.best_params_))

    file.write("\n\nBest Macro F1 Cross-Validation:\n")
    file.write(str(round(random_search.best_score_, 4)))


print("\nHasil tuning disimpan di:")
print(result_path)

print("\nCATATAN:")
print(
    "test_final_locked.csv TIDAK digunakan "
    "pada proses tuning ini."
)